# Polymarket + Firecrawl + Mistral Workflow

This notebook walks through a small single-agent workflow step by step:

1. fetch a Polymarket market
2. collect a few public web sources with Firecrawl
3. ask Mistral Large for a simulated trade decision
4. inspect the structured JSON output

No trades are placed here — this is research and simulation only.

In [1]:
from __future__ import annotations

import json
import os
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Optional

import pandas as pd
import requests
from dotenv import load_dotenv
from IPython.display import display, Markdown


In [2]:
# Load local environment variables from .env if present

def load_local_env(env_path: str = '.env') -> None:
    path = Path(env_path)
    if path.exists():
        load_dotenv(path)

load_local_env()

GAMMA_BASE = 'https://gamma-api.polymarket.com'
FIRECRAWL_SEARCH_URL = 'https://api.firecrawl.dev/v1/search'
MISTRAL_API_URL = 'https://api.mistral.ai/v1/chat/completions'

# Easy-to-edit knobs for development
MARKET_LIMIT = 10
MARKET_ORDER_BY = 'volume24hr'
MARKET_ASCENDING = 'false'
MARKET_ACTIVE = 'true'
MARKET_CLOSED = 'false'
FIRECRAWL_LIMIT = 5

print('Environment loaded.')
print('MISTRAL_API_KEY set:', bool(os.getenv('MISTRAL_API_KEY')))
print('FIRECRAWL_API_KEY set:', bool(os.getenv('FIRECRAWL_API_KEY')))

Environment loaded.
MISTRAL_API_KEY set: True
FIRECRAWL_API_KEY set: True


In [3]:
@dataclass
class MarketCandidate:
    id: str
    question: str
    slug: str | None = None
    volume24hr: float | None = None
    liquidity: float | None = None
    outcome_prices: list[float] | None = None
    clob_token_ids: list[str] | None = None


def parse_jsonish(value: Any) -> Any:
    if isinstance(value, str):
        try:
            return json.loads(value)
        except json.JSONDecodeError:
            return value
    return value


def safe_float(value: Any) -> Optional[float]:
    try:
        if value is None or value == '':
            return None
        return float(value)
    except (TypeError, ValueError):
        return None


def parse_token_ids(raw: Any) -> list[str]:
    parsed = parse_jsonish(raw)
    if isinstance(parsed, list):
        return [str(item) for item in parsed if str(item).strip()]
    if isinstance(parsed, str):
        return [part.strip() for part in parsed.split(',') if part.strip()]
    return []

print('Helpers ready.')

Helpers ready.


In [4]:
session = requests.Session()
print('Session created for reusable HTTP calls.')

Session created for reusable HTTP calls.


In [5]:
def ask_llm(
    mistral_api_key: str,
    model: str,
    prompt: str,
    response_format: dict[str, Any] | None = None,
) -> dict[str, Any]:
    request_body: dict[str, Any] = {
        'model': model,
        'messages': [{'role': 'user', 'content': prompt}],
        'temperature': 0.2,
    }
    if response_format is not None:
        request_body['response_format'] = response_format
    else:
        request_body['response_format'] = {'type': 'json_object'}

    response = requests.post(
        MISTRAL_API_URL,
        headers={
            'Authorization': f'Bearer {mistral_api_key}',
            'Content-Type': 'application/json',
        },
        json=request_body,
        timeout=90,
    )
    response.raise_for_status()
    payload = response.json()
    content = payload['choices'][0]['message']['content'] or ''
    match = re.search(r'```(?:json)?\s*([\s\S]*?)\s*```', content)
    json_text = match.group(1) if match else content
    return json.loads(json_text)


In [6]:
def fetch_markets(limit: int = MARKET_LIMIT) -> list[dict[str, Any]]:
    params = {
        'limit': limit,
        'order': MARKET_ORDER_BY,
        'ascending': MARKET_ASCENDING,
        'active': MARKET_ACTIVE,
        'closed': MARKET_CLOSED,
    }
    response = session.get(f'{GAMMA_BASE}/markets', params=params, timeout=30)
    response.raise_for_status()
    data = response.json()
    return data if isinstance(data, list) else []

markets = fetch_markets(limit=MARKET_LIMIT)
print(f'Fetched {len(markets)} markets.')
market_df = pd.DataFrame(markets)
display(market_df[['id', 'question', 'slug', 'volume24hr', 'liquidity']].head(5))

Fetched 10 markets.


,id,question,slug,volume24hr,liquidity
0,1706358,"Military action against Iran ends by April 17,...",military-action-against-iran-ends-by-april-17-...,1.309859e+07,5972888.24848
1,1800030,Will Manchester United FC win on 2026-04-13?,epl-mun-lee-2026-04-13-mun,3.031052e+06,1363879.80141
2,1501211,Will La U win the third most seats in the 2026...,will-la-u-win-the-third-most-seats-in-the-2026...,2.807746e+06,6668.73758
3,567561,Will the next Prime Minister of Hungary be Pét...,will-the-next-prime-minister-of-hungary-be-pte...,2.517417e+06,628235.31538
4,947289,Will Roberto Sánchez Palomino win the 2026 Per...,will-roberto-snchez-palomino-win-the-2026-peru...,2.047950e+06,119278.1899


In [7]:
def choose_market(markets: list[dict[str, Any]]) -> MarketCandidate:
    if not markets:
        raise RuntimeError('No Polymarket markets were returned.')
    market = markets[0] # For simplicity, just take the first market. You can implement more complex logic here. TODO
    return MarketCandidate(
        id=str(market.get('id', '')),
        question=str(market.get('question', '')),
        slug=market.get('slug'),
        volume24hr=safe_float(market.get('volume24hr')),
        liquidity=safe_float(market.get('liquidity')),
        outcome_prices=parse_jsonish(market.get('outcomePrices')),
        clob_token_ids=parse_token_ids(market.get('clobTokenIds')),
    )

selected_market = choose_market(markets)
print('Selected market:')
display(pd.DataFrame([selected_market.__dict__]))

Selected market:


,id,question,slug,volume24hr,liquidity,outcome_prices,clob_token_ids
0,1706358,"Military action against Iran ends by April 17,...",military-action-against-iran-ends-by-april-17-...,1.309859e+07,5.972888e+06,"[0.9995, 0.0005]",[110577858780148651266447864913251141042726823...


In [8]:
def fetch_firecrawl_context(query: str) -> list[dict[str, Any]]:
    firecrawl_api_key = os.getenv('FIRECRAWL_API_KEY')
    if not firecrawl_api_key:
        return []

    headers = {
        'Authorization': f'Bearer {firecrawl_api_key}',
        'Content-Type': 'application/json',
    }
    payload = {
        'query': query,
        'limit': FIRECRAWL_LIMIT,
        'scrapeOptions': {'formats': ['markdown'], 'onlyMainContent': True},
    }
    response = session.post(FIRECRAWL_SEARCH_URL, headers=headers, json=payload, timeout=60)
    response.raise_for_status()
    data = response.json()
    results = data.get('data', []) if isinstance(data, dict) else []
    return results if isinstance(results, list) else []

sources = fetch_firecrawl_context(selected_market.question)
if not sources:
    sources = fetch_firecrawl_context(f'"{selected_market.question}"')

print(f'Collected {len(sources)} Firecrawl sources.')
sources_df = pd.DataFrame([{
    'title': s.get('title') or s.get('metadata', {}).get('title'),
    'url': s.get('url') or s.get('link'),
    'snippet': (s.get('markdown') or s.get('content') or s.get('description') or '')[:250],
} for s in sources])
display(sources_df.head(5))

Collected 5 Firecrawl sources.


,title,url,snippet
0,2026 Iran war - Wikipedia,https://en.wikipedia.org/wiki/2026_Iran_war,[Jump to content](https://en.wikipedia.org/wik...
1,"Day 39 of Middle East conflict — US, Israel, I...",https://www.cnn.com/2026/04/07/world/live-news...,- [![Shipping in and around the Strait of Horm...
2,"2026 Iran war | Explained, United States, Isra...",https://www.britannica.com/event/2026-Iran-war,[Ask the Chatbot](https://www.britannica.com/c...
3,"On The Hour – April 12, 2026 | Ceasefire Crack...",https://www.youtube.com/watch?v=CCzNYryYzg0,Error 403 (Forbidden)!!1\n\n**403.** That’s an...
4,"Deadline passes for US blockade of Hormuz, Ira...",https://www.youtube.com/watch?v=YXjzUp62Ddk,Error 403 (Forbidden)!!1\n\n**403.** That’s an...


In [9]:
def build_prompt(market: MarketCandidate, sources: list[dict[str, Any]]) -> str:
    source_text = []
    for item in sources[:5]:
        title = item.get('title') or item.get('metadata', {}).get('title') or 'Untitled'
        url = item.get('url') or item.get('link') or ''
        snippet = item.get('markdown') or item.get('content') or item.get('description') or ''
        source_text.append(f'- {title}\n  URL: {url}\n  Snippet: {snippet[:1200]}')

    sources_block = '\n\n'.join(source_text) if source_text else 'No external sources were collected.'
    token_ids = ', '.join(market.clob_token_ids or []) or 'None found'

    return f'''
You are a Polymarket research agent.

Your job is to read one market, inspect a few public web sources, and make a simulated trading decision.
Do not place any trades. Return only valid JSON.

Market:
- question: {market.question}
- slug: {market.slug}
- volume24hr: {market.volume24hr}
- liquidity: {market.liquidity}
- outcome_prices: {market.outcome_prices}
- clob_token_ids: {token_ids}

Web context:
{sources_block}

Decision rules:
- Prefer markets where the event is well-defined, time-bounded, and researchable from public sources.
- If the evidence is weak or the market is too ambiguous, choose skip.
- If you choose a side, use 'YES' or 'NO'.
- Include a short thesis and mention the most important source(s).

Return JSON with these fields:
{{
  "decision": "buy_yes" | "buy_no" | "skip",
  "confidence": 0.0,
  "side": "YES" | "NO" | null,
  "entry_price": 0.0 | null,
  "fair_value": 0.0 | null,
  "expected_value_edge": 0.0 | null,
  "reasoning": ["..."],
  "key_sources": ["..."],
  "risk_flags": ["..."],
  "research_summary": "..."
}}
'''.strip()



if not os.getenv('MISTRAL_API_KEY'):
    raise RuntimeError('MISTRAL_API_KEY is not set.')

structured_output_schema = {
    'type': 'json_schema',
    'json_schema': {
        'name': 'polymarket_trade_decision',
        'strict': True,
        'schema': {
            'type': 'object',
            'additionalProperties': False,
            'properties': {
                'decision': {'type': 'string', 'enum': ['buy_yes', 'buy_no', 'skip']},
                'confidence': {'type': 'number'},
                'side': {'type': ['string', 'null'], 'enum': ['YES', 'NO', None]},
                'entry_price': {'type': ['number', 'null']},
                'fair_value': {'type': ['number', 'null']},
                'expected_value_edge': {'type': ['number', 'null']},
                'reasoning': {'type': 'array', 'items': {'type': 'string'}},
                'key_sources': {'type': 'array', 'items': {'type': 'string'}},
                'risk_flags': {'type': 'array', 'items': {'type': 'string'}},
                'research_summary': {'type': 'string'},
            },
            'required': [
                'decision',
                'confidence',
                'side',
                'entry_price',
                'fair_value',
                'expected_value_edge',
                'reasoning',
                'key_sources',
                'risk_flags',
                'research_summary',
            ],
        },
    },
}

prompt = build_prompt(selected_market, sources)
print(prompt[:3000])


You are a Polymarket research agent.

Your job is to read one market, inspect a few public web sources, and make a simulated trading decision.
Do not place any trades. Return only valid JSON.

Market:
- question: Military action against Iran ends by April 17, 2026?
- slug: military-action-against-iran-ends-by-april-17-2026
- volume24hr: 13098587.00157
- liquidity: 5972888.24848
- outcome_prices: ['0.9995', '0.0005']
- clob_token_ids: 110577858780148651266447864913251141042726823916797235908223754289720625159449, 72231738155593996978526441927387175895154152018539080152695312064797491581743

Web context:
- 2026 Iran war - Wikipedia
  URL: https://en.wikipedia.org/wiki/2026_Iran_war
  Snippet: [Jump to content](https://en.wikipedia.org/wiki/2026_Iran_war#bodyContent)

[![Extended-protected article](https://upload.wikimedia.org/wikipedia/en/thumb/8/8c/Extended-protection-shackle.svg/20px-Extended-protection-shackle.svg.png)](https://en.wikipedia.org/wiki/Wikipedia:Protection_policy#extende

In [10]:
raw_decision = ask_llm(
    os.getenv('MISTRAL_API_KEY', ''),
    os.getenv('MISTRAL_MODEL', 'mistral-large-2512'),
    prompt,
    response_format=structured_output_schema,
)
print(json.dumps(raw_decision, indent=2, ensure_ascii=False))

display(Markdown('### Parsed decision summary'))
display(pd.DataFrame([raw_decision]))


{
  "decision": "skip",
  "confidence": 0.0,
  "side": null,
  "entry_price": null,
  "fair_value": null,
  "expected_value_edge": null,
  "reasoning": [
    "The market question is highly speculative and time-bounded (April 17, 2026), but the provided web context is contradictory and unreliable.",
    "Key sources (e.g., Wikipedia, CNN, Britannica) suggest an *ongoing* conflict as of April 2026, with no clear resolution in sight. For example, CNN reports a ceasefire agreement on April 7, 2026, but later notes that ceasefire talks collapsed by April 12, 2026 (YouTube snippet).",
    "The Wikipedia page is marked as 'extended-protected' and rapidly changing, indicating instability in the narrative. Britannica provides a historical overview but no definitive endpoint for the conflict.",
    "YouTube snippets reference escalating tensions (e.g., US blockade of Hormuz, Iranian threats), but the videos themselves are inaccessible (403 errors), limiting verification.",
    "The current Polym

### Parsed decision summary

,decision,confidence,side,entry_price,fair_value,expected_value_edge,reasoning,key_sources,risk_flags,research_summary
0,skip,0.0,None,None,None,None,[The market question is highly speculative and...,"[https://en.wikipedia.org/wiki/2026_Iran_war, ...",[Rapidly evolving geopolitical situation with ...,The market asks whether military action agains...
